In [31]:
import json

from dotenv import find_dotenv, load_dotenv
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser
from langchain.schema.runnable import RunnableMap
from langchain.vectorstores import DocArrayInMemorySearch
from langchain_ollama import ChatOllama, OllamaEmbeddings

In [2]:
load_dotenv(find_dotenv("../../creds/.env"), verbose=True)

True

In [3]:
llm = ChatOllama(
    base_url="http://localhost:11434",
    model="qwen3:4B",
    temperature=0,
    verbose=True,
    extract_reasoning=True,
)

embedding = OllamaEmbeddings(model="nomic-embed-text", base_url="http://localhost:11434")

### Simple Chain

In [4]:
prompt = ChatPromptTemplate.from_template("tell me a short joke about {topic} in {time}")
output_parser = StrOutputParser()

In [5]:
chain = prompt | llm | output_parser

In [6]:
chain.invoke({"topic": "fish", "time": "medieval"})

'\n\n**Joke:**  \nWhy did the fish go to the medieval castle?  \nTo *fish* for a crown! 🐟👑  \n\n*(A play on "fish" as both a verb and noun, with a medieval twist!)*'

### More complex chain

In [7]:
vectorstore = DocArrayInMemorySearch.from_texts(
    ["harrison worked at kensho", "bears like to eat honey"],
    embedding=embedding,
)
retriever = vectorstore.as_retriever()

/home/eugene/projects/deeplearning.ai/.venv/lib/python3.13/site-packages/docarray/helper.py:255: SyntaxWarning: invalid escape sequence '\*'
  e.g. '\*.py', '[\*.zip, \*.gz]'
/home/eugene/projects/deeplearning.ai/.venv/lib/python3.13/site-packages/pydantic/_migration.py:283: UserWarning: `pydantic.error_wrappers:ValidationError` has been moved to `pydantic:ValidationError`.
  warnings.warn(f'`{import_path}` has been moved to `{new_location}`.')


In [8]:
retriever.get_relevant_documents("where did harrison work?")

/tmp/ipykernel_44012/3310280720.py:1: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  retriever.get_relevant_documents("where did harrison work?")


[Document(metadata={}, page_content='harrison worked at kensho'),
 Document(metadata={}, page_content='bears like to eat honey')]

In [9]:
template = """
Answer the question based only on the following context: {context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\nAnswer the question based only on the following context: {context}\n\nQuestion: {question}\n'), additional_kwargs={})])

In [10]:
runnable_map = RunnableMap(
    {
        "context": lambda x: retriever.get_relevant_documents(x["question"]),
        "question": lambda x: x["question"],
    }
)

chain = runnable_map | prompt | llm | output_parser
chain

{
  context: RunnableLambda(...),
  question: RunnableLambda(...)
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\nAnswer the question based only on the following context: {context}\n\nQuestion: {question}\n'), additional_kwargs={})])
| ChatOllama(model='qwen3:4B', extract_reasoning=True, temperature=0.0, base_url='http://localhost:11434')
| StrOutputParser()

In [11]:
chain.invoke({"question": "where did harrison work?"})

'\n\nBased on the provided context, Harrison worked at **Kensho**.'

In [12]:
runnable_map.invoke({"question": "where did harrison work?"})

{'context': [Document(metadata={}, page_content='harrison worked at kensho'),
  Document(metadata={}, page_content='bears like to eat honey')],
 'question': 'where did harrison work?'}

### Bind

In [25]:
functions = [
    {
        "name": "weather_search",
        "description": "Search for weather given an airport code",
        "parameters": {
            "type": "object",
            "properties": {
                "airport_code": {"type": "string", "description": "The airport code to get the weather for"},
            },
            "required": ["airport_code"],
        },
    },
    {
        "name": "sports_search",
        "description": "Search for news of recent sport events",
        "parameters": {
            "type": "object",
            "properties": {
                "team_name": {"type": "string", "description": "The sports team to search for"},
            },
            "required": ["team_name"],
        },
    },
]

In [26]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
    ]
)
model = ChatOllama(
    base_url="http://localhost:11434",
    model="qwen3:4B",
    temperature=0,
    verbose=True,
    extract_reasoning=True,
).bind_tools(
    tools=functions,
)

runnable = prompt | model

In [27]:
runnable.invoke({"input": "what is the weather in Kyiv?"})

AIMessage(content='\n\n', additional_kwargs={'reasoning_content': "<think>\nOkay, the user is asking for the weather in Kyiv. Let me check the tools available. There's a weather_search function that requires an airport code. Kyiv's airport code is KIV. So I need to call weather_search with airport_code set to KIV. I should make sure that's the correct code. Alternatively, maybe the user meant another airport, but Kyiv's main airport is indeed KIV. So the function call should be straightforward.\n</think>"}, response_metadata={'model': 'qwen3:4B', 'created_at': '2025-06-25T17:06:21.863951919Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2018854920, 'load_duration': 39819203, 'prompt_eval_count': 212, 'prompt_eval_duration': 71409732, 'eval_count': 120, 'eval_duration': 1906450910, 'model_name': 'qwen3:4B'}, id='run--caa87296-6182-478f-8bac-adfa25ff5f80-0', tool_calls=[{'name': 'weather_search', 'args': {'airport_code': 'KIV'}, 'id': 'c697176c-3965-4582-99be-42b57f8a2fd0', 't

In [29]:
runnable.invoke({"input": "what was the last game Dynamo Kyiv won?"})

AIMessage(content='\n\n', additional_kwargs={'reasoning_content': '<think>\nOkay, the user is asking about the last game that Dynamo Kyiv won. Let me see. The available tools are weather_search and sports_search. The weather_search function takes an airport code, which isn\'t relevant here. The sports_search function requires a team name. Dynamo Kyiv is a sports team, so I should use the sports_search function. The parameter needed is team_name, so I\'ll set that to "Dynamo Kyiv". That should retrieve the recent sports news or game results for them. Let me make sure I\'m using the right function. Yes, sports_search is the correct one here. I\'ll call that with the team name.\n</think>'}, response_metadata={'model': 'qwen3:4B', 'created_at': '2025-06-25T17:09:27.864463049Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2583493680, 'load_duration': 26928029, 'prompt_eval_count': 214, 'prompt_eval_duration': 26851629, 'eval_count': 161, 'eval_duration': 2528713848, 'model_name':

### Fallbacks

In [43]:
simple_chain = llm | json.loads
complex_chain = llm | StrOutputParser() | json.loads

In [44]:
challenge = "write three poems in a VALID json blob, where each poem is a json blob of a title, author, and first line"

In [45]:
simple_chain.invoke(challenge)

TypeError: the JSON object must be str, bytes or bytearray, not AIMessage

In [47]:
final_chain = simple_chain.with_fallbacks([complex_chain])
final_chain.invoke(challenge)

{'poems': [{'title': 'Whispers of the Willow',
   'author': 'Eleanor Frost',
   'first line': 'Beneath the boughs where shadows dance'},
  {'title': 'The Rose and the Moon',
   'author': 'Lila Marlow',
   'first line': 'In the hush between the stars and the earth'},
  {'title': 'The Clockwork Heart',
   'author': 'Raphael Vane',
   'first line': 'Time is a river, but the heart is a stone'}]}

### Interface

In [50]:
prompt = ChatPromptTemplate.from_template("Tell me a short but conventionally funny joke about {topic}")

chain = prompt | llm | StrOutputParser()

In [51]:
chain.invoke({"topic": "piranha"})

'\n\n**Why did the piranha get kicked out of the pool?**  \nBecause it was always *biting* the toes of the other fish! 🐠😄  \n\n*(A classic play on the aggressive nature of piranhas and a pun on "biting the toes" leading to a humorous consequence.)*'

In [53]:
chain.batch([{"topic": "goose"}, {"topic": "music"}])

['\n\n**Why don\'t geese ever get cold?**  \n*They always have a lot of feathers.*  \n\n(Play on the double meaning of "feathers" as both the physical down and the phrase "a lot of feathers" implying warmth.)',
 '\n\n**Why don\'t musicians ever get cold?**  \n*They always have a warm note.* 🎶  \n\n*(A classic pun on "warm note" and the idea of staying cozy in music!)*']

In [56]:
for t in chain.stream({"topic": "bears"}):
    if t.strip():
        print(t)

Why
 don
't
 bears
 ever
 get
 cold
?
Because
 they
 have
 a
 lot
 of
 *
fur
*
!
*(
Alternatively
,
 the
 classic
 pun
:
)*
What
 do
 you
 call
 a
 bear
 with
 no
 teeth
?
A
 *
g
ummy
 bear
*
!
 🐻
🍬


In [57]:
response = await chain.ainvoke({"topic": "bears"})
response

"\n\nWhy don't bears ever get cold?  \nBecause they have a lot of *fur*!  \n\n*(Alternatively, the classic pun:)*  \nWhat do you call a bear with no teeth?  \nA *gummy bear*! 🐻🍬"